# Batch Normalization — Implementation / 구현

**Paper**: Ioffe, S. & Szegedy, C. (2015). "Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift." ICML 2015.

이 노트북은 Batch Normalization의 핵심 알고리즘을 처음부터 구현하고, 논문의 주요 실험 결과를 재현합니다.

This notebook implements Batch Normalization from scratch and reproduces the paper's key experimental findings.

**구현 내용 / Contents:**
1. **Part 1**: BN Transform forward & backward (Algorithm 1) — NumPy로 처음부터 구현
2. **Part 2**: BN을 포함한 네트워크로 MNIST 훈련 — PyTorch로 논문 Figure 1 재현
3. **Part 3**: Activation 분포 시각화 — BN 유무에 따른 internal covariate shift 비교
4. **Part 4**: Learning rate sensitivity 실험 — BN이 높은 learning rate를 허용하는지 검증

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## Part 1: BN Transform from Scratch (NumPy) / BN 변환 처음부터 구현

논문의 Algorithm 1을 NumPy로 구현합니다. Forward pass와 backward pass를 모두 포함합니다.

We implement Algorithm 1 from the paper using NumPy, including both forward and backward passes.

### Algorithm 1 (논문 p.3):
- **Input**: Mini-batch $\mathcal{B} = \{x_{1...m}\}$; learnable parameters $\gamma$, $\beta$
- **Output**: $\{y_i = \text{BN}_{\gamma,\beta}(x_i)\}$

$$\mu_\mathcal{B} \leftarrow \frac{1}{m}\sum_{i=1}^m x_i, \quad
\sigma_\mathcal{B}^2 \leftarrow \frac{1}{m}\sum_{i=1}^m (x_i - \mu_\mathcal{B})^2$$

$$\hat{x}_i \leftarrow \frac{x_i - \mu_\mathcal{B}}{\sqrt{\sigma_\mathcal{B}^2 + \epsilon}}, \quad
y_i \leftarrow \gamma\hat{x}_i + \beta$$

In [ ]:
class BatchNormNumPy:
    """Batch Normalization from scratch (Algorithm 1 in the paper).

    Implements forward and backward pass exactly as described in Ioffe &
    Szegedy (2015), Section 3.  Supports both training mode (mini-batch
    statistics) and inference mode (running / population statistics).
    """

    def __init__(self, num_features: int, eps: float = 1e-5,
                 momentum: float = 0.1):
        self.eps = eps
        self.momentum = momentum

        # Learnable parameters (gamma, beta)
        self.gamma = np.ones(num_features)
        self.beta = np.zeros(num_features)

        # Running statistics for inference (Algorithm 2, lines 8-11)
        self.running_mean = np.zeros(num_features)
        self.running_var = np.ones(num_features)

        # Cache for backward pass
        self.cache = None

    def forward(self, x: np.ndarray, training: bool = True) -> np.ndarray:
        """Forward pass of BN Transform.

        Args:
            x: Input of shape (m, d) where m is batch size, d is features.
            training: If True, use mini-batch statistics; else population stats.

        Returns:
            y: BN-transformed output of shape (m, d).
        """
        if training:
            # Step 1: mini-batch mean
            mu_B = np.mean(x, axis=0)                          # (d,)

            # Step 2: mini-batch variance
            sigma_B_sq = np.var(x, axis=0)                     # (d,)

            # Step 3: normalize
            x_hat = (x - mu_B) / np.sqrt(sigma_B_sq + self.eps)  # (m, d)

            # Step 4: scale and shift
            y = self.gamma * x_hat + self.beta                 # (m, d)

            # Update running statistics (exponential moving average)
            self.running_mean = ((1 - self.momentum) * self.running_mean
                                 + self.momentum * mu_B)
            self.running_var = ((1 - self.momentum) * self.running_var
                                + self.momentum * sigma_B_sq)

            # Cache for backward
            self.cache = (x, x_hat, mu_B, sigma_B_sq)
        else:
            # Inference: use population statistics (Algorithm 2, line 11)
            x_hat = ((x - self.running_mean)
                     / np.sqrt(self.running_var + self.eps))
            y = self.gamma * x_hat + self.beta

        return y

    def backward(self, dy: np.ndarray) -> np.ndarray:
        """Backward pass through BN (paper p.4, before simplification).

        Args:
            dy: Gradient of loss w.r.t. output y, shape (m, d).

        Returns:
            dx: Gradient of loss w.r.t. input x, shape (m, d).
        """
        x, x_hat, mu_B, sigma_B_sq = self.cache
        m = x.shape[0]
        inv_std = 1.0 / np.sqrt(sigma_B_sq + self.eps)

        # Gradients w.r.t. gamma and beta
        self.dgamma = np.sum(dy * x_hat, axis=0)       # (d,)
        self.dbeta = np.sum(dy, axis=0)                 # (d,)

        # Gradient w.r.t. x_hat
        dx_hat = dy * self.gamma                        # (m, d)

        # Gradient w.r.t. sigma_B^2
        dsigma_sq = np.sum(
            dx_hat * (x - mu_B) * (-0.5) * inv_std**3, axis=0
        )                                               # (d,)

        # Gradient w.r.t. mu_B
        dmu = (np.sum(dx_hat * (-inv_std), axis=0)
               + dsigma_sq * np.mean(-2.0 * (x - mu_B), axis=0))  # (d,)

        # Gradient w.r.t. x
        dx = (dx_hat * inv_std
              + dsigma_sq * 2.0 * (x - mu_B) / m
              + dmu / m)                                # (m, d)

        return dx


# --- Verify implementation ---
print("=== BatchNorm Forward/Backward Verification ===\n")

bn = BatchNormNumPy(num_features=4)
x = np.random.randn(8, 4) * 3 + 5  # batch=8, features=4, mean~5, std~3

y = bn.forward(x, training=True)
print(f"Input  mean: {x.mean(axis=0).round(3)}")
print(f"Input  std:  {x.std(axis=0).round(3)}")
print(f"Output mean: {y.mean(axis=0).round(3)}  (should be ~0)")
print(f"Output std:  {y.std(axis=0).round(3)}  (should be ~1)")

# Verify backward with numerical gradient
dy = np.random.randn(*y.shape)
dx_analytic = bn.backward(dy)

dx_numeric = np.zeros_like(x)
h = 1e-5
for i in range(x.shape[0]):
    for j in range(x.shape[1]):
        x_plus = x.copy(); x_plus[i, j] += h
        x_minus = x.copy(); x_minus[i, j] -= h
        y_plus = bn.forward(x_plus, training=True)
        y_minus = bn.forward(x_minus, training=True)
        dx_numeric[i, j] = np.sum((y_plus - y_minus) * dy) / (2 * h)

max_err = np.max(np.abs(dx_analytic - dx_numeric))
print(f"\nBackward gradient check — max error: {max_err:.2e} (should be < 1e-5)")

### Identity Recovery Test / 항등 변환 복원 테스트

논문 p.3에서 언급: $\gamma = \sqrt{\text{Var}[x]}$, $\beta = \text{E}[x]$로 설정하면 원래 activation을 완벽히 복원할 수 있다.

As noted on p.3: setting $\gamma = \sqrt{\text{Var}[x]}$ and $\beta = \text{E}[x]$ perfectly recovers the original activations.

In [ ]:
# Identity recovery: gamma = std(x), beta = mean(x) should recover x
bn_identity = BatchNormNumPy(num_features=4)
x_test = np.random.randn(32, 4) * 7 - 3  # arbitrary mean and std

bn_identity.gamma = np.std(x_test, axis=0)
bn_identity.beta = np.mean(x_test, axis=0)

y_recovered = bn_identity.forward(x_test, training=True)
max_recovery_err = np.max(np.abs(x_test - y_recovered))
print(f"Identity recovery max error: {max_recovery_err:.2e} (should be ~0)")
print(f"This confirms BN never limits representational power.")

## Part 2: MNIST Training — BN vs No-BN (논문 Figure 1 재현) / Reproducing Figure 1

논문의 MNIST 실험을 재현합니다: 3개의 fully-connected hidden layer (각 100 units), sigmoid nonlinearity.
BN 유무에 따른 test accuracy와 activation 분포 변화를 비교합니다.

We reproduce the paper's MNIST experiment: 3 fully-connected hidden layers (100 units each) with sigmoid nonlinearity.
We compare test accuracy and activation distribution changes with and without BN.

In [ ]:
# Load MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))  # Flatten 28x28 -> 784
])

train_dataset = datasets.MNIST(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = datasets.MNIST(
    root='./data', train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=60, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

print(f"Train samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")

In [ ]:
class MNISTNet(nn.Module):
    """MNIST network matching the paper's Section 4.1 setup.

    3 fully-connected hidden layers with 100 units each, sigmoid
    nonlinearity, and optional Batch Normalization.
    """

    def __init__(self, use_bn: bool = False):
        super().__init__()
        self.use_bn = use_bn

        self.fc1 = nn.Linear(784, 100)
        self.fc2 = nn.Linear(100, 100)
        self.fc3 = nn.Linear(100, 100)
        self.fc_out = nn.Linear(100, 10)

        if use_bn:
            self.bn1 = nn.BatchNorm1d(100)
            self.bn2 = nn.BatchNorm1d(100)
            self.bn3 = nn.BatchNorm1d(100)

        # Initialize weights to small random Gaussian (as in the paper)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)

        # Storage for activation statistics
        self.activation_history = {
            'layer3_mean': [], 'layer3_std': [],
            'layer3_percentiles': []
        }

    def forward(self, x, record_stats: bool = False):
        """Forward pass with optional activation recording."""
        if self.use_bn:
            h1 = torch.sigmoid(self.bn1(self.fc1(x)))
            h2 = torch.sigmoid(self.bn2(self.fc2(h1)))
            h3_pre = self.bn3(self.fc3(h2))
        else:
            h1 = torch.sigmoid(self.fc1(x))
            h2 = torch.sigmoid(self.fc2(h1))
            h3_pre = self.fc3(h2)

        if record_stats:
            with torch.no_grad():
                vals = h3_pre[:, 0].cpu().numpy()  # Track 1st activation
                self.activation_history['layer3_mean'].append(vals.mean())
                self.activation_history['layer3_std'].append(vals.std())
                pcts = np.percentile(vals, [15, 50, 85])
                self.activation_history['layer3_percentiles'].append(pcts)

        h3 = torch.sigmoid(h3_pre)
        out = self.fc_out(h3)
        return out


def evaluate(model, loader):
    """Compute accuracy on a dataset."""
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total


def train_model(model, train_loader, test_loader, num_steps=10000,
                lr=0.01, eval_every=500):
    """Train a model and record accuracy history."""
    model = model.to(device)
    optimizer = optim.SGD(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    step = 0
    accuracies = []
    steps_record = []
    train_iter = iter(train_loader)

    while step < num_steps:
        model.train()
        try:
            x, y = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            x, y = next(train_iter)

        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x, record_stats=(step % 100 == 0))
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        step += 1

        if step % eval_every == 0:
            acc = evaluate(model, test_loader)
            accuracies.append(acc)
            steps_record.append(step)
            print(f"  Step {step:>6d} | Test accuracy: {acc:.4f}")

    return steps_record, accuracies


print("Model architecture (with BN):")
print(MNISTNet(use_bn=True))

In [ ]:
# Train both models (paper uses 50000 steps, we use 10000 for speed)
NUM_STEPS = 10000

print("=" * 50)
print("Training WITHOUT Batch Normalization")
print("=" * 50)
torch.manual_seed(42)
model_no_bn = MNISTNet(use_bn=False)
steps_no_bn, acc_no_bn = train_model(
    model_no_bn, train_loader, test_loader, num_steps=NUM_STEPS
)

print("\n" + "=" * 50)
print("Training WITH Batch Normalization")
print("=" * 50)
torch.manual_seed(42)
model_bn = MNISTNet(use_bn=True)
steps_bn, acc_bn = train_model(
    model_bn, train_loader, test_loader, num_steps=NUM_STEPS
)

In [ ]:
# --- Figure 1(a): Test accuracy comparison (reproducing paper Figure 1a) ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) Accuracy curves
axes[0].plot(steps_no_bn, acc_no_bn, 'b-', label='Without BN', linewidth=2)
axes[0].plot(steps_bn, acc_bn, 'r-', label='With BN', linewidth=2)
axes[0].set_xlabel('Training Steps')
axes[0].set_ylabel('Test Accuracy')
axes[0].set_title('(a) Test Accuracy — BN vs No-BN\n(cf. Paper Figure 1a)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# (b) Activation distribution WITHOUT BN (cf. Paper Figure 1b)
hist_no_bn = model_no_bn.activation_history
pcts_no_bn = np.array(hist_no_bn['layer3_percentiles'])
x_axis = np.arange(len(pcts_no_bn)) * 100

axes[1].plot(x_axis, pcts_no_bn[:, 0], 'b--', alpha=0.7, label='15th pct')
axes[1].plot(x_axis, pcts_no_bn[:, 1], 'b-', linewidth=2, label='50th pct (median)')
axes[1].plot(x_axis, pcts_no_bn[:, 2], 'b--', alpha=0.7, label='85th pct')
axes[1].fill_between(x_axis, pcts_no_bn[:, 0], pcts_no_bn[:, 2], alpha=0.15, color='blue')
axes[1].set_xlabel('Training Steps')
axes[1].set_ylabel('Activation Value (pre-sigmoid)')
axes[1].set_title('(b) Without BN — Layer 3 Activation\n(cf. Paper Figure 1b)')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

# (c) Activation distribution WITH BN (cf. Paper Figure 1c)
hist_bn = model_bn.activation_history
pcts_bn = np.array(hist_bn['layer3_percentiles'])
x_axis_bn = np.arange(len(pcts_bn)) * 100

axes[2].plot(x_axis_bn, pcts_bn[:, 0], 'r--', alpha=0.7, label='15th pct')
axes[2].plot(x_axis_bn, pcts_bn[:, 1], 'r-', linewidth=2, label='50th pct (median)')
axes[2].plot(x_axis_bn, pcts_bn[:, 2], 'r--', alpha=0.7, label='85th pct')
axes[2].fill_between(x_axis_bn, pcts_bn[:, 0], pcts_bn[:, 2], alpha=0.15, color='red')
axes[2].set_xlabel('Training Steps')
axes[2].set_ylabel('Activation Value (pre-sigmoid)')
axes[2].set_title('(c) With BN — Layer 3 Activation\n(cf. Paper Figure 1c)')
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('19_ioffe_2015_figure1.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n→ BN network trains faster and achieves higher accuracy (Figure 1a).")
print("→ Without BN, activation distributions shift significantly (Figure 1b).")
print("→ With BN, activation distributions remain stable (Figure 1c).")

## Part 3: Scale Invariance Property / 스케일 불변성 검증

논문 Section 3.3의 핵심 성질을 검증합니다:

We verify the key property from Section 3.3:

$$\text{BN}(Wu) = \text{BN}((aW)u) \quad \forall a > 0$$

Weight의 scale이 BN 출력에 영향을 주지 않으며, 큰 weight는 작은 gradient를 받아 자동 안정화됨.

Weight scale does not affect BN output, and larger weights receive smaller gradients for automatic stabilization.

In [ ]:
# Demonstrate scale invariance: BN(Wu) == BN((aW)u)
W = np.random.randn(5, 3)   # weight matrix
u = np.random.randn(16, 5)  # input batch

scale_factors = [0.01, 0.1, 1.0, 10.0, 100.0]

print("=== Scale Invariance: BN(Wu) == BN((aW)u) ===\n")
print(f"{'Scale a':>10s} | {'max|BN(Wu) - BN(aWu)|':>25s} | {'||grad_W||':>12s} | {'||grad_aW||':>12s} | {'ratio':>8s}")
print("-" * 80)

# Reference: BN(Wu) with a=1
x_ref = u @ W
bn_ref = BatchNormNumPy(num_features=3)
y_ref = bn_ref.forward(x_ref, training=True)

for a in scale_factors:
    aW = a * W
    x_scaled = u @ aW

    bn_scaled = BatchNormNumPy(num_features=3)
    y_scaled = bn_scaled.forward(x_scaled, training=True)

    # Check output equality
    output_diff = np.max(np.abs(y_ref - y_scaled))

    # Check gradient scaling: d(BN)/d(aW) = (1/a) * d(BN)/dW
    dy = np.random.randn(*y_ref.shape)

    bn_ref_grad = BatchNormNumPy(num_features=3)
    bn_ref_grad.forward(x_ref, training=True)
    dx_ref = bn_ref_grad.backward(dy)
    grad_W = u.T @ dx_ref  # gradient w.r.t. W

    bn_scaled_grad = BatchNormNumPy(num_features=3)
    bn_scaled_grad.forward(x_scaled, training=True)
    dx_scaled = bn_scaled_grad.backward(dy)
    grad_aW = u.T @ dx_scaled  # gradient w.r.t. aW

    grad_W_norm = np.linalg.norm(grad_W)
    grad_aW_norm = np.linalg.norm(grad_aW)
    ratio = grad_aW_norm / grad_W_norm if grad_W_norm > 0 else float('inf')

    print(f"{a:>10.2f} | {output_diff:>25.2e} | {grad_W_norm:>12.6f} | {grad_aW_norm:>12.6f} | {ratio:>8.4f}")

print("\n→ Output is identical regardless of weight scale (output_diff ≈ 0)")
print("→ Gradient norm scales as 1/a — larger weights get smaller gradients")

## Part 4: Learning Rate Sensitivity / 학습률 민감도 실험

논문의 핵심 결과 중 하나: BN이 높은 learning rate를 허용한다 (Section 3.3, Figure 2-3).
BN 유무에 따라 다양한 learning rate에서의 훈련 안정성을 비교합니다.

One of the paper's key findings: BN enables higher learning rates (Section 3.3, Figures 2-3).
We compare training stability across various learning rates with and without BN.

In [ ]:
# Compare learning rate sensitivity
learning_rates = [0.001, 0.01, 0.1, 0.5]
LR_STEPS = 5000

results_no_bn = {}
results_bn = {}

for lr in learning_rates:
    print(f"\n--- Learning rate: {lr} ---")

    # Without BN
    torch.manual_seed(42)
    model = MNISTNet(use_bn=False)
    try:
        steps, accs = train_model(
            model, train_loader, test_loader,
            num_steps=LR_STEPS, lr=lr, eval_every=1000
        )
        results_no_bn[lr] = (steps, accs)
    except Exception as e:
        print(f"  No-BN FAILED at lr={lr}: {e}")
        results_no_bn[lr] = None

    # With BN
    torch.manual_seed(42)
    model = MNISTNet(use_bn=True)
    try:
        steps, accs = train_model(
            model, train_loader, test_loader,
            num_steps=LR_STEPS, lr=lr, eval_every=1000
        )
        results_bn[lr] = (steps, accs)
    except Exception as e:
        print(f"  BN FAILED at lr={lr}: {e}")
        results_bn[lr] = None

In [ ]:
# Plot learning rate sensitivity comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for ax, results, title in [
    (axes[0], results_no_bn, 'Without BN'),
    (axes[1], results_bn, 'With BN')
]:
    for (lr, data), color in zip(results.items(), colors):
        if data is not None:
            steps, accs = data
            ax.plot(steps, accs, '-o', color=color, label=f'lr={lr}',
                    linewidth=2, markersize=4)
            final_acc = accs[-1] if accs else 0
            ax.annotate(f'{final_acc:.1%}', xy=(steps[-1], accs[-1]),
                       fontsize=9, ha='left')
        else:
            ax.axhline(y=0.1, color=color, linestyle=':', alpha=0.5,
                       label=f'lr={lr} (diverged)')
    ax.set_xlabel('Training Steps')
    ax.set_ylabel('Test Accuracy')
    ax.set_title(f'{title} — Learning Rate Sensitivity')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1.0])

plt.tight_layout()
plt.savefig('19_ioffe_2015_lr_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print("\n=== Final Accuracy Summary ===\n")
print(f"{'LR':>8s} | {'Without BN':>12s} | {'With BN':>12s}")
print("-" * 40)
for lr in learning_rates:
    no_bn_acc = f"{results_no_bn[lr][1][-1]:.1%}" if results_no_bn[lr] else "diverged"
    bn_acc = f"{results_bn[lr][1][-1]:.1%}" if results_bn[lr] else "diverged"
    print(f"{lr:>8.3f} | {no_bn_acc:>12s} | {bn_acc:>12s}")

print("\n→ BN maintains stable training across a much wider range of learning rates")
print("→ Without BN, high learning rates cause training instability or failure")

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| **Batch Normalization** | 미니배치 통계로 정규화 + learnable $\gamma$, $\beta$ | `nn.BatchNorm1d/2d` — 거의 모든 CNN의 기본 |
| **BN backward** | 6개 chain-rule 도함수 수동 유도 | PyTorch autograd가 자동 처리 |
| **Inference mode** | Moving average로 population stats 추정 | `model.eval()` — 자동 전환 |
| **Scale invariance** | $\text{BN}(aWu) = \text{BN}(Wu)$ | Weight normalization 등 관련 기법도 존재 |
| **Regularization 효과** | BN의 stochastic noise → Dropout 대체 | 현대 모델에서 BN + Dropout 동시 사용은 드묾 |
| **CNN BN** | Per-feature-map normalization | GroupNorm, LayerNorm 등 대안 활발 |

### 핵심 교훈 / Key Lessons

1. **BN은 단순하지만 혁명적**: 4줄의 수식(Algorithm 1)이 딥러닝 훈련 방식을 근본적으로 바꿨다.
   BN is simple but revolutionary: 4 lines of math (Algorithm 1) fundamentally changed deep learning training.

2. **"정규화를 네트워크 안에 포함시켜라"**: 외부 정규화는 gradient descent와 충돌한다. 네트워크 아키텍처의 일부로 만들어야 gradient가 정규화를 인식한다.
   "Embed normalization inside the network": external normalization conflicts with gradient descent. Making it part of the architecture ensures gradients account for it.

3. **$\gamma$, $\beta$의 설계가 핵심**: 정규화만으로는 표현력을 제한한다. Learnable affine transform이 표현력을 보존하면서 정규화의 이점을 유지한다.
   The design of $\gamma$, $\beta$ is key: normalization alone limits representational power. Learnable affine transform preserves capacity while maintaining normalization benefits.